# 09 - Yeni dış test veri seti araştırması

Bu notebook'un amacı, mevcut Reuters/SP500 deneylerinden bağımsız bir dış test seti seçmek ve gerekiyorsa küçük ama savunulabilir bir insan anotasyonu süreci tasarlamaktır.

Araştırma tarihi: 2026-08-03

Karar özeti:
- Birincil öneri: SEntFiN 1.0 / v1.1. İnsan etiketli, finans haber başlığı odaklı, entity-aware yapısı var ve mevcut Reuters/SP500 setinden daha bağımsız görünüyor.
- İkinci öneri: FiQA 2018. Akademik olarak güçlü ve küçük; ancak yaygın benchmark olduğu için FinBERT/finans modelleriyle dolaylı veri aşinalığı riski daha yüksek.
- Elle etiketleme havuzu: NOSIBLE Financial Sentiment veya güncel finans haber başlığı kaynakları. Büyük ve güncel, fakat LLM/model etiketli olduğu için doğrudan gold test seti olarak değil, örnek seçip insan doğrulama için kullanılmalı.
- Mevcut veriyle çakıştığı için zayıf aday: S&P 500 with Financial News Headlines (2008-2024). Projede zaten SP500 haber başlığı verisi kullanıldığı için yeni dış test katkısı sınırlı.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
APP_DIR_CANDIDATES = [ROOT / "app", ROOT]
for app_dir_candidate in APP_DIR_CANDIDATES:
    if (app_dir_candidate / "thesis_utils").exists() and str(app_dir_candidate) not in sys.path:
        sys.path.insert(0, str(app_dir_candidate))

try:
    from thesis_utils.paths import PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, PROCESSED_DATA_DIR, EVALUATION_DATA_DIR, ensure_data_directories
except Exception as e:
    raise RuntimeError(f"Yol import edilemedi: {e}") from e

ensure_data_directories()

EXTERNAL_CANDIDATE_DIR = DATA_ROOT / "external_candidates"
EXTERNAL_CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", ROOT)
print("Project root:", PROJECT_ROOT)
print("External candidate dir:", EXTERNAL_CANDIDATE_DIR)


Working directory: C:\MyRepo\software\financial_sentiment_thesis\app
Project root: C:\MyRepo\software\financial_sentiment_thesis
External candidate dir: C:\MyRepo\software\financial_sentiment_thesis\db\external_candidates


## Kaynak notları

Kullanılan güncel kaynaklar ve gerekçeler:

- SEntFiN 1.0 / Kaggle v1.1: 10.700+ finans haber başlığı, entity-sentiment anotasyonu, 2.800+ çoklu entity içeren başlık ve JASIST makalesiyle akademik dayanak. Kaynak: https://www.kaggle.com/datasets/ankurzing/aspect-based-sentiment-analysis-for-financial-news
- FiQA 2018: Aspect-based financial sentiment task; skorlar -1 ile +1 aralığında, haber başlığı ve microblog örnekleri içeriyor, non-commercial kullanım notu var. Kaynak: https://sites.google.com/view/fiqa/home ve Hugging Face aynası: https://huggingface.co/datasets/TheFinAI/fiqa-sentiment-classification
- NOSIBLE Financial Sentiment: 100.000 temizlenmiş finans haber/snippet örneği, positive/neutral/negative etiketi, URL ve domain alanları var; etiketler LLM tabanlı pipeline ile üretilmiş. Kaynak: https://huggingface.co/datasets/NOSIBLE/financial-sentiment
- SENTiVENT Bench: ekonomik olaylar ve örtük finansal sentiment için uzman anotasyonlu yeni benchmark; yapılandırılmış/event-level değerlendirme için değerli, doğrudan 3 sınıflı plain sentiment testine dönüştürmek ek eşleme gerektirir. Kaynak: https://huggingface.co/datasets/GillesJacobs/sentivent
- S&P 500 with Financial News Headlines (2008-2024): 19.000+ başlık ve kapanış fiyatı; ancak bu projede SP500 tabanlı veri zaten kullanıldığı için bağımsız dış test olarak uygun değil. Kaynak: https://www.kaggle.com/datasets/dyutidasmahaptra/s-and-p-500-with-financial-news-headlines-20082024


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

candidate_datasets = [
    {
        "rank": 1,
        "name": "SEntFiN 1.0 / v1.1",
        "source_url": "https://www.kaggle.com/datasets/ankurzing/aspect-based-sentiment-analysis-for-financial-news",
        "label_type": "human entity-sentiment labels",
        "size_note": "10,700+ headlines; 2,800+ multi-entity headlines",
        "fit_for_plain_sentiment": 8,
        "fit_for_target_sentiment": 10,
        "academic_strength": 9,
        "novelty_vs_project": 9,
        "manual_annotation_need": "low: labels exist; only mapping/spot-check needed",
        "decision": "primary_external_test",
        "notes": "Best candidate because it is finance-news-specific, human annotated, and entity-aware. For plain sentiment, collapse entity labels to headline-level with a documented rule.",
    },
    {
        "rank": 2,
        "name": "FiQA 2018 sentiment classification",
        "source_url": "https://huggingface.co/datasets/TheFinAI/fiqa-sentiment-classification",
        "label_type": "continuous score from -1 to 1; target/aspect fields",
        "size_note": "1,173 examples in common HF mirror",
        "fit_for_plain_sentiment": 8,
        "fit_for_target_sentiment": 9,
        "academic_strength": 10,
        "novelty_vs_project": 8,
        "manual_annotation_need": "low: map score thresholds to negative/neutral/positive",
        "decision": "secondary_external_test",
        "notes": "Very defensible benchmark. Main risk: it is widely used in finance NLP, so model pretraining or prior evaluation leakage should be discussed.",
    },
    {
        "rank": 3,
        "name": "NOSIBLE Financial Sentiment",
        "source_url": "https://huggingface.co/datasets/NOSIBLE/financial-sentiment",
        "label_type": "LLM/active-learning sentiment labels with source URLs",
        "size_note": "100,000 examples",
        "fit_for_plain_sentiment": 9,
        "fit_for_target_sentiment": 5,
        "academic_strength": 6,
        "novelty_vs_project": 10,
        "manual_annotation_need": "medium/high: sample 300-600 examples and human-validate labels",
        "decision": "manual_annotation_pool",
        "notes": "Strong pool for a new manually checked test set. Do not use raw labels as gold without validation because labels are generated by an LLM pipeline.",
    },
    {
        "rank": 4,
        "name": "SENTiVENT Bench",
        "source_url": "https://huggingface.co/datasets/GillesJacobs/sentivent",
        "label_type": "expert event/implicit sentiment annotations",
        "size_note": "financial-news event benchmark",
        "fit_for_plain_sentiment": 6,
        "fit_for_target_sentiment": 8,
        "academic_strength": 8,
        "novelty_vs_project": 10,
        "manual_annotation_need": "medium: map structured annotations to the project sentiment schema",
        "decision": "advanced_follow_up",
        "notes": "Promising for a richer evaluation chapter, but the label schema is more complex than the current 3-class sentiment setup.",
    },
    {
        "rank": 5,
        "name": "S&P 500 with Financial News Headlines (2008-2024)",
        "source_url": "https://www.kaggle.com/datasets/dyutidasmahaptra/s-and-p-500-with-financial-news-headlines-20082024",
        "label_type": "headline + market close; no direct human sentiment labels",
        "size_note": "19,000+ headlines",
        "fit_for_plain_sentiment": 5,
        "fit_for_target_sentiment": 3,
        "academic_strength": 5,
        "novelty_vs_project": 2,
        "manual_annotation_need": "high: already close to current SP500 data, would need new sampling and human labels",
        "decision": "reject_as_primary",
        "notes": "Useful as background, but weak as a new independent test set because the project already uses SP500 headline data.",
    },
]

comparison_df = pd.DataFrame(candidate_datasets)
comparison_df["weighted_score"] = (
    comparison_df["fit_for_plain_sentiment"] * 0.25
    + comparison_df["fit_for_target_sentiment"] * 0.25
    + comparison_df["academic_strength"] * 0.25
    + comparison_df["novelty_vs_project"] * 0.25
).round(2)
comparison_df = comparison_df.sort_values(["weighted_score", "rank"], ascending=[False, True]).reset_index(drop=True)

display(Markdown("## Aday veri seti karar matrisi"))
display(comparison_df[["rank", "name", "weighted_score", "decision", "label_type", "manual_annotation_need", "notes"]])

comparison_df.to_csv(EXTERNAL_CANDIDATE_DIR / "candidate_dataset_research_matrix.csv", index=False)
print("Kaydedildi:", EXTERNAL_CANDIDATE_DIR / "candidate_dataset_research_matrix.csv")


ModuleNotFoundError: No module named 'pandas'

In [ ]:
recommendation = """
Önerilen yol:

1. Ana dış test seti olarak SEntFiN 1.0 / v1.1 kullanılmalı.
   - Target-level modeller için doğrudan entity-sentiment değerlendirmesi yapılabilir.
   - Plain sentiment modeller için entity etiketleri headline-level etikete dönüştürülür.

2. İkinci kontrol seti olarak FiQA 2018 kullanılmalı.
   - Sürekli sentiment skoru üç sınıfa çevrilir: score <= -0.1 negative, -0.1 < score < 0.1 neutral, score >= 0.1 positive.
   - Tezde benchmark aşinalığı/leakage riski not edilir.

3. Yeni ve tez için daha özgün bir mini test seti gerekiyorsa NOSIBLE veya güncel finans haberlerinden 300-600 örnek seçilip elle etiketlenmeli.
   - Sınıf dengesi hedefi: en az 100 negative, 100 neutral, 100 positive.
   - Her örnek iki kişi tarafından etiketlenmeli; anlaşmazlıklar üçüncü turda çözülmeli.
"""

display(Markdown("## Son karar"))
print(recommendation)


In [ ]:
annotation_columns = [
    "sample_id",
    "source_dataset",
    "source_url",
    "date",
    "text",
    "target_entity",
    "candidate_label",
    "annotator_1_label",
    "annotator_2_label",
    "final_label",
    "disagreement_flag",
    "notes",
]

annotation_template = pd.DataFrame(columns=annotation_columns)
template_path = EXTERNAL_CANDIDATE_DIR / "manual_annotation_template.csv"
annotation_template.to_csv(template_path, index=False)

guideline_rows = [
    {"label": "positive", "rule": "Haber, yatırımcı açısından hedef şirket/piyasa için olumlu beklenti veya gerçekleşme ifade eder."},
    {"label": "negative", "rule": "Haber, yatırımcı açısından hedef şirket/piyasa için olumsuz beklenti, risk veya kayıp ifade eder."},
    {"label": "neutral", "rule": "Haber çoğunlukla olgusal, belirsiz veya sentiment yönü net olmayan bilgi verir."},
    {"label": "conflict", "rule": "Birden fazla hedef için farklı sentiment varsa target_entity doldurulur; plain sentiment için örnek ayrılır veya final_label boş bırakılır."},
]
guideline_df = pd.DataFrame(guideline_rows)
guideline_path = EXTERNAL_CANDIDATE_DIR / "manual_annotation_guidelines.csv"
guideline_df.to_csv(guideline_path, index=False)

display(Markdown("## Elle etiketleme şablonu"))
display(annotation_template)
display(guideline_df)
print("Şablon kaydedildi:", template_path)
print("Kılavuz kaydedildi:", guideline_path)


## Mevcut DB veri setlerinden örnekler

Aşağıdaki hücre, hocaya veri setlerinin niteliğini göstermek için ana `db` kaynaklarının her birinden 5'er örnek satır üretir. Batch Excel dosyaları tek tek listelenmez; onların yerine master/anotasyon dosyaları gösterilir.


In [ ]:
from IPython.display import Markdown, display
import pandas as pd

local_dataset_specs = [
    {
        "dataset": "Raw SP500 headlines",
        "path": DATA_ROOT / "raw" / "sp500_headlines_2008_2024_raw.csv",
        "purpose": "Ham finans haber başlıkları; mevcut SP500 tabanlı ana kaynak.",
    },
    {
        "dataset": "Aggregated financial news enriched",
        "path": DATA_ROOT / "processed" / "aggregated_financial_news_enriched.csv",
        "purpose": "Birleştirilmiş ve zenginleştirilmiş haber verisi; modelleme öncesi geniş kaynak.",
    },
    {
        "dataset": "SP500 FinBERT labeled headlines",
        "path": DATA_ROOT / "processed" / "sp500_headlines_2008_2024_finbert_labeled.csv",
        "purpose": "FinBERT ile otomatik etiketlenmiş SP500 başlıkları; pseudo-label karakterini göstermek için.",
    },
    {
        "dataset": "Market sentiment modeling master",
        "path": DATA_ROOT / "processed" / "market_sentiment_modeling_master.parquet",
        "purpose": "Piyasa yönü/sentiment modellemesi için hazırlanmış ana modelleme tablosu.",
    },
    {
        "dataset": "Plain sentiment training dataset",
        "path": DATA_ROOT / "processed" / "training_datasets" / "plain_sentiment_dataset.csv",
        "purpose": "3 sınıflı plain sentiment eğitim veri seti.",
    },
    {
        "dataset": "Target-level sentiment training dataset",
        "path": DATA_ROOT / "processed" / "training_datasets" / "target_level_sentiment_dataset.csv",
        "purpose": "Hedef şirket/entity odaklı sentiment eğitim veri seti.",
    },
    {
        "dataset": "SP500 human annotation master",
        "path": DATA_ROOT / "annotations" / "sp500_balanced_1500" / "sp500_balanced_1500_annotation_master.csv",
        "purpose": "Dengeli seçilmiş SP500 insan anotasyon master seti.",
    },
    {
        "dataset": "Reuters human annotation master",
        "path": DATA_ROOT / "annotations" / "reuters_5000" / "reuters_annotation_5000_master.csv",
        "purpose": "Reuters 5000 insan anotasyon master seti.",
    },
    {
        "dataset": "Pseudo-labeled confidence 0.90 news",
        "path": DATA_ROOT / "interim" / "pseudo_labeled_news_confidence_090.parquet",
        "purpose": "Yüksek güvenli otomatik etiketlenmiş ara veri.",
    },
    {
        "dataset": "Plain sentiment train split",
        "path": DATA_ROOT / "splits" / "plain_sentiment_v1" / "train_df.parquet",
        "purpose": "Mevcut plain sentiment model eğitimi için train bölmesi.",
    },
    {
        "dataset": "Plain sentiment validation split",
        "path": DATA_ROOT / "splits" / "plain_sentiment_v1" / "val_df.parquet",
        "purpose": "Mevcut plain sentiment model seçimi için validation bölmesi.",
    },
    {
        "dataset": "Plain sentiment test split",
        "path": DATA_ROOT / "splits" / "plain_sentiment_v1" / "test_df.parquet",
        "purpose": "Mevcut plain sentiment model değerlendirme bölmesi.",
    },
    {
        "dataset": "Synthetic evaluation negative examples",
        "path": DATA_ROOT / "evaluation" / "synthetic_financial_news" / "negative_batch_000.xlsx",
        "purpose": "Sentetik değerlendirme setinin negative sınıfından örnek batch.",
    },
    {
        "dataset": "Synthetic evaluation neutral examples",
        "path": DATA_ROOT / "evaluation" / "synthetic_financial_news" / "neutral_batch_001.xlsx",
        "purpose": "Sentetik değerlendirme setinin neutral sınıfından örnek batch.",
    },
    {
        "dataset": "Synthetic evaluation positive examples",
        "path": DATA_ROOT / "evaluation" / "synthetic_financial_news" / "positive_batch_000.xlsx",
        "purpose": "Sentetik değerlendirme setinin positive sınıfından örnek batch.",
    },
]

preferred_column_keywords = [
    "id",
    "date",
    "source",
    "ticker",
    "company",
    "target",
    "entity",
    "headline",
    "title",
    "text",
    "sentence",
    "label",
    "sentiment",
    "score",
    "confidence",
]

def read_preview(path, n=5):
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Desteklenmeyen dosya tipi: {path.suffix}")

    selected_columns = []
    lower_columns = {column.lower(): column for column in df.columns}
    for keyword in preferred_column_keywords:
        for lower_column, original_column in lower_columns.items():
            if keyword in lower_column and original_column not in selected_columns:
                selected_columns.append(original_column)

    if not selected_columns:
        selected_columns = list(df.columns[:8])
    else:
        selected_columns = selected_columns[:8]

    preview = df[selected_columns].dropna(how="all")
    if len(preview) > n:
        preview = preview.sample(n=n, random_state=42).sort_index()
    return df, preview.reset_index(drop=True)

all_examples = []
summary_rows = []

display(Markdown("## Ana DB veri setlerinden 5'er örnek"))

for spec in local_dataset_specs:
    path = spec["path"]
    display(Markdown(f"### {spec['dataset']}"))
    display(Markdown(f"{spec['purpose']}<br>`{path.relative_to(PROJECT_ROOT)}`"))

    if not path.exists():
        display(Markdown("Dosya bulunamadı."))
        summary_rows.append({**spec, "exists": False, "rows": None, "columns": None})
        continue

    df, preview = read_preview(path)
    display(preview)

    exported_preview = preview.copy()
    exported_preview.insert(0, "dataset", spec["dataset"])
    exported_preview.insert(1, "purpose", spec["purpose"])
    all_examples.append(exported_preview)

    summary_rows.append(
        {
            "dataset": spec["dataset"],
            "path": str(path.relative_to(PROJECT_ROOT)),
            "purpose": spec["purpose"],
            "exists": True,
            "rows": len(df),
            "columns": len(df.columns),
        }
    )

dataset_summary_df = pd.DataFrame(summary_rows)
display(Markdown("## Örneklenen veri setleri özeti"))
display(dataset_summary_df)

if all_examples:
    examples_path = EXTERNAL_CANDIDATE_DIR / "local_dataset_examples_for_presentation.csv"
    pd.concat(all_examples, ignore_index=True, sort=False).to_csv(examples_path, index=False)
    dataset_summary_df.to_csv(EXTERNAL_CANDIDATE_DIR / "local_dataset_examples_summary.csv", index=False)
    print("Sunum örnekleri kaydedildi:", examples_path)
    print("Özet kaydedildi:", EXTERNAL_CANDIDATE_DIR / "local_dataset_examples_summary.csv")


## Sonraki çalışma adımları

1. SEntFiN CSV dosyasını `db/raw/external/sentfin/` altına indir.
2. Entity-level etiketleri projenin `negative`, `neutral`, `positive` şemasına dönüştüren küçük bir hazırlık notebook'u veya hücresi ekle.
3. Aynı metnin birden fazla entity için farklı etiketi varsa iki değerlendirme oluştur: target-level test ve conflict-free plain sentiment test.
4. FiQA için skor eşiklerini sabitle ve `db/evaluation/fiqa2018/` altında model-ready CSV üret.
5. Ek özgün katkı gerekiyorsa NOSIBLE'dan dengeli 300-600 örnek seç, `manual_annotation_template.csv` ile iki kişi tarafından etiketle ve Cohen's kappa raporla.
